# 🧠 Single Agent Pipeline Project

## What This Is About
We're building a simple assistant that can understand what someone's
asking, decide which tool fits the job, use that tool, and reply back
in a clean, consistent format.

### It should be able to handle:
- Math questions → hand off to the Calculator
- Requests to pull out keywords → hand off to the Keyword Extractor
- Anything else → just reply normally

---
### 🛠️ What Needed to Be Built
- The agent's decision-making logic
- Routing based on what the query contains
- Hooking up the tools
- Handling errors gracefully instead of crashing

### 🚀 Extras Added
- A smarter fallback response
- Simple logging so you can see which route was picked
- A third tool for general queries


In [1]:
# Tool 1: Calculator
# This just does basic math. If someone types a bad expression
# (like dividing by zero, or random text), we don't want the whole
# program to crash, so we catch the error and send back a simple message instead.

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [2]:
# Tool 2: Keyword Extractor
# Pulls out up to 5 "important" words from a sentence.
# Keeping it simple here: a word counts as a keyword if it's longer than
# 4 letters. It's a rough filter, not real NLP, but it's enough for this project.

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

In [3]:
# Tool 3 : General Response
# This is the fallback for anything that isn't a math or keyword request.
# Instead of just ignoring the query, we acknowledge it so the user
# knows the agent actually understood what they typed.

def general_response(query: str) -> dict:
    """Fallback handler for queries that don't match a specific tool."""
    try:
        return {
            "type": "general",
            "result": f"I don't have a specific tool for this, but here's an acknowledgment of your query: '{query.strip()}'"
        }
    except Exception:
        return {"type": "error", "result": "Could not process general query."}

## 🤖 Building the Agent's Logic

The routing rule is simple:
- Query mentions "calculate" → send it to the calculator
- Query mentions "keywords" → send it to the keyword extractor
- Anything else → just give a general reply

In [4]:
# The Agent
# This is the brain of the whole thing. It looks at what the user typed,
# figures out which tool should handle it, and always hands back an answer
# in the same {"type": ..., "result": ...} shape, no matter what happens.

def agent(query: str, verbose: bool = True):
    """
    Looks at the query, decides which tool should handle it,
    calls that tool, and returns a clean, structured result.
    Never crashes, even if something goes wrong along the way.
    """

    # Nothing to work with, so just say so instead of guessing.
    if not isinstance(query, str) or not query.strip():
        return {"type": "error", "result": "Empty or invalid query."}

    query_lower = query.lower()

    try:
        # Case 1: looks like a math request
        if "calculate" in query_lower:
            if verbose:
                print("[router] matched -> calculator tool")

            # Remove the word "calculate" so we're left with just the actual expression
            expression = query_lower.replace("calculate", "", 1).strip()

            if not expression:
                return {"type": "error", "result": "No expression found after 'calculate'."}

            result = calculator(expression)

            if result == "Error in calculation":
                return {"type": "error", "result": f"Could not evaluate expression: '{expression}'"}

            return {"type": "calculation", "result": result}

        # Case 2: looks like a keyword extraction request
        elif "keywords" in query_lower:
            if verbose:
                print("[router] matched -> keyword extractor tool")

            keywords = extract_keywords(query)

            if not keywords:
                return {"type": "error", "result": "No keywords could be extracted."}

            return {"type": "keywords", "result": keywords}

        # Case 3: doesn't match anything specific, so just acknowledge it
        else:
            if verbose:
                print("[router] matched -> general response")

            return general_response(query)

    except Exception as e:
        # Safety net: whatever goes wrong, the agent still replies cleanly
        # instead of crashing the program (this matters most in the interactive loop below).
        if verbose:
            print(f"[router] error -> {e}")
        return {"type": "error", "result": f"Unexpected error: {str(e)}"}

## 📦 What the Output Looks Like

Every response follows this same shape, no matter which tool handled it:

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [5]:
# Trying it out
# A quick run through a few normal queries plus a few "what if this goes wrong"
# cases, just to make sure the agent handles both the easy stuff and the edge cases.

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?",
    "Calculate",          # what if there's no actual math after the word "calculate"?
    "Calculate 10 / 0",   # what if the math itself is invalid?
    "",                   # what if someone submits nothing at all?
]

for q in queries:
    print("Query:", repr(q))
    print("Response:", agent(q))
    print("-" * 50)

Query: 'Calculate 20 + 5'
[router] matched -> calculator tool
Response: {'type': 'calculation', 'result': '25'}
--------------------------------------------------
Query: 'Extract keywords from Artificial Intelligence is transforming industries'
[router] matched -> keyword extractor tool
Response: {'type': 'keywords', 'result': ['industries', 'intelligence', 'artificial', 'transforming', 'extract']}
--------------------------------------------------
Query: 'What is machine learning?'
[router] matched -> general response
Response: {'type': 'general', 'result': "I don't have a specific tool for this, but here's an acknowledgment of your query: 'What is machine learning?'"}
--------------------------------------------------
Query: 'Calculate'
[router] matched -> calculator tool
Response: {'type': 'error', 'result': "No expression found after 'calculate'."}
--------------------------------------------------
Query: 'Calculate 10 / 0'
[router] matched -> calculator tool
Response: {'type': 'er

In [6]:
# Try it yourself
# Type any query below and the agent will route it live.
# Type 'exit' whenever you want to stop.

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))

Enter query (type 'exit' to stop): Calculate 45 * 3
[router] matched -> calculator tool
Response: {'type': 'calculation', 'result': '135'}
Enter query (type 'exit' to stop): Extract keywords from Machine learning models require large datasets
[router] matched -> keyword extractor tool
Response: {'type': 'keywords', 'result': ['learning', 'datasets', 'require', 'large', 'models']}
Enter query (type 'exit' to stop): What is deep learning?
[router] matched -> general response
Response: {'type': 'general', 'result': "I don't have a specific tool for this, but here's an acknowledgment of your query: 'What is deep learning?'"}
Enter query (type 'exit' to stop): Calculate 5 / 0
[router] matched -> calculator tool
Response: {'type': 'error', 'result': "Could not evaluate expression: '5 / 0'"}
Enter query (type 'exit' to stop): exit
